<a href="https://colab.research.google.com/github/maxGrigorenko/DL_HSE/blob/hw_7/ZeroS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Григоренко Максим. ZeroS Linear Attention на примере задачи классфикации отзывов

In [ ]:
!git clone https://github.com/LJC-FVNR/SequenceLab.git
%cd SequenceLab
!pip install -e .
!pip install datasets transformers

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

# Загружаем классический датасет длинных текстов
dataset = load_dataset("stanfordnlp/imdb")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Длинный контекст
MAX_LEN = 1024

def tokenize_long_text(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LEN)

print("Токенизируем длинные тексты (это займет около минуты)...")
tokenized_imdb = dataset.map(tokenize_long_text, batched=True)
tokenized_imdb.set_format(type="torch", columns=["input_ids", "label"])

BATCH_SIZE = 16

train_loader = torch.utils.data.DataLoader(tokenized_imdb["train"], batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(tokenized_imdb["test"], batch_size=BATCH_SIZE, shuffle=False)

print(f"Размер батча: {BATCH_SIZE}, Длина последовательности: {MAX_LEN}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

Токенизируем длинные тексты (это займет около минуты)...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Размер батча: 16, Длина последовательности: 1024


In [3]:
NUM_CLASSES = 2
EMBED_DIM = 128

In [4]:
import torch.nn as nn
from sequencelab.build import build_attention
from sequencelab.config import ZeroSConfig

class ZeroSClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, seq_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # ZeroS слой для обработки текстовой последовательности
        cfg = ZeroSConfig(
            n_embd=embed_dim,
            n_head=4,
            block_size=seq_len,
            is_causal=False,
            use_norm=True
        )
        self.zeros_layer = build_attention(cfg)

        # Классификатор: берем среднее по последовательности (pooling)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)      # [B, T, D]
        x = self.zeros_layer(x)    # [B, T, D]
        x = x.mean(dim=1)          # Global Average Pooling
        return self.classifier(x)  # [B, Num_Classes]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ZeroSClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=EMBED_DIM,
    num_classes=NUM_CLASSES,
    seq_len=MAX_LEN
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [5]:
from tqdm.auto import tqdm

EPOCHS = 3

print(f"Запускаем обучение ZeroS на {device}...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total_samples = 0

    progress_bar = tqdm(train_loader, desc=f"Эпоха {epoch+1}/{EPOCHS}")
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        # Forward pass (ZeroS)
        logits = model(input_ids)

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        # метрики
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        progress_bar.set_postfix({
            'loss': f"{total_loss / (progress_bar.n + 1):.4f}",
            'acc': f"{correct / total_samples:.4f}"
        })

    epoch_loss = total_loss / len(train_loader)
    epoch_acc = correct / total_samples * 100
    print(f"Итоги эпохи {epoch+1} | Средний Loss: {epoch_loss:.4f} | Точность (Accuracy): {epoch_acc:.2f}%\n")

Запускаем обучение ZeroS на cuda...


Эпоха 1/3:   0%|          | 0/1563 [00:00<?, ?it/s]

Итоги эпохи 1 | Средний Loss: 0.4886 | Точность (Accuracy): 75.67%



Эпоха 2/3:   0%|          | 0/1563 [00:00<?, ?it/s]

Итоги эпохи 2 | Средний Loss: 0.2751 | Точность (Accuracy): 88.94%



Эпоха 3/3:   0%|          | 0/1563 [00:00<?, ?it/s]

Итоги эпохи 3 | Средний Loss: 0.1800 | Точность (Accuracy): 93.24%



Сравним со стандратным softmax

In [6]:
import torch.optim as optim

class SoftmaxClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # Классическое Softmax-внимание (квадратичное)
        self.attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=4, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)

        # Self-attention: Query, Key, Value равны x
        attn_out, _ = self.attention(x, x, x)
        x = self.norm(x + attn_out)

        # Global Average Pooling
        x = x.mean(dim=1)
        return self.classifier(x)


baseline_model = SoftmaxClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=EMBED_DIM,
    num_classes=NUM_CLASSES
).to(device)

# Те же loss и оптимизатор, что и у ZeroS, для сравнения
criterion = nn.CrossEntropyLoss()
optimizer_base = optim.AdamW(baseline_model.parameters(), lr=1e-3)

EPOCHS = 3

print(f"Запускаем обучение бейзлайна (Softmax) на {device}...")

for epoch in range(EPOCHS):
    baseline_model.train()
    total_loss = 0
    correct = 0
    total_samples = 0

    # Тот же прогресс-бар для удобства
    progress_bar = tqdm(train_loader, desc=f"Эпоха {epoch+1}/{EPOCHS} (Softmax)")

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer_base.zero_grad()

        # Forward pass классического внимания
        logits = baseline_model(input_ids)

        # Вычисляем ошибку
        loss = criterion(logits, labels)

        # Backward pass
        loss.backward()
        optimizer_base.step()

        # Собираем метрики
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        # Обновляем прогресс-бар
        progress_bar.set_postfix({
            'loss': f"{total_loss / (progress_bar.n + 1):.4f}",
            'acc': f"{correct / total_samples:.4f}"
        })

    epoch_loss = total_loss / len(train_loader)
    epoch_acc = correct / total_samples * 100
    print(f"Итоги эпохи {epoch+1} (Softmax) | Средний Loss: {epoch_loss:.4f} | Точность (Accuracy): {epoch_acc:.2f}%\n")

Запускаем обучение бейзлайна (Softmax) на cuda...


Эпоха 1/3 (Softmax):   0%|          | 0/1563 [00:00<?, ?it/s]

Итоги эпохи 1 (Softmax) | Средний Loss: 0.5073 | Точность (Accuracy): 74.86%



Эпоха 2/3 (Softmax):   0%|          | 0/1563 [00:00<?, ?it/s]

Итоги эпохи 2 (Softmax) | Средний Loss: 0.3631 | Точность (Accuracy): 84.21%



Эпоха 3/3 (Softmax):   0%|          | 0/1563 [00:00<?, ?it/s]

Итоги эпохи 3 (Softmax) | Средний Loss: 0.2635 | Точность (Accuracy): 89.14%



In [7]:
import torch.nn.functional as F

# Классическое линейное внимание (Katharopoulos Linear Attention)
class StandardLinearAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def feature_map(self, x):
        # Гарантируем строго положительные значения: elu(x) + 1
        return F.elu(x) + 1.0

    def forward(self, x):
        B, T, C = x.size()

        # Получаем Q, K, V
        qkv = self.qkv_proj(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2] # [B, num_heads, T, head_dim]

        # Применяем feature map
        q = self.feature_map(q)
        k = self.feature_map(k)

        # O(N): Сначала умножаем K на V, а только потом на Q
        # kv_state: [B, num_heads, head_dim, head_dim]
        kv_state = torch.matmul(k.transpose(-2, -1), v)

        # Считаем знаменатель для нормализации (сумма по K)
        k_sum = k.sum(dim=-2, keepdim=True)
        z = 1.0 / (torch.matmul(q, k_sum.transpose(-2, -1)) + 1e-6)

        # Получаем итоговое внимание
        out = torch.matmul(q, kv_state) * z

        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)

class KatharopoulosClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = StandardLinearAttention(embed_dim, num_heads=4)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        attn_out = self.attention(x)
        x = self.norm(x + attn_out)
        x = x.mean(dim=1)
        return self.classifier(x)

# Инициализируем и обучаем линейный бейзлайн (те же 3 эпохи)
linear_model = KatharopoulosClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=EMBED_DIM,
    num_classes=NUM_CLASSES
).to(device)

optimizer_linear = torch.optim.AdamW(linear_model.parameters(), lr=1e-3)

print(f"Запускаем обучение линейного бейзлайна (Katharopoulos) на {device}...")
for epoch in range(EPOCHS):
    linear_model.train()
    for batch in tqdm(train_loader, desc=f"Эпоха {epoch+1}/{EPOCHS} (Linear)"):
        input_ids, labels = batch["input_ids"].to(device), batch["label"].to(device)
        optimizer_linear.zero_grad()
        loss = criterion(linear_model(input_ids), labels)
        loss.backward()
        optimizer_linear.step()

Запускаем обучение линейного бейзлайна (Katharopoulos) на cuda...


Эпоха 1/3 (Linear):   0%|          | 0/1563 [00:00<?, ?it/s]

Эпоха 2/3 (Linear):   0%|          | 0/1563 [00:00<?, ?it/s]

Эпоха 3/3 (Linear):   0%|          | 0/1563 [00:00<?, ?it/s]

In [8]:
# ФИНАЛЬНЫЙ ТЕСТ ВСЕХ ТРЕХ АРХИТЕКТУР
def evaluate_model(eval_model, name):
    eval_model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Тестируем {name}"):
            input_ids, labels = batch["input_ids"].to(device), batch["label"].to(device)
            logits = eval_model(input_ids)
            preds = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print(f"[{name}] Точность на тестовой выборке: {correct / total * 100:.2f}%\n")

print("\n--- ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ ---")
# Передаем модели, которые обучили в предыдущих ячейках, и новую линейную
evaluate_model(model, "ZeroS (Zero-Sum Linear)")
evaluate_model(baseline_model, "Softmax (Quadratic Baseline)")
evaluate_model(linear_model, "Katharopoulos (Positive Linear)")


--- ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ ---


Тестируем ZeroS (Zero-Sum Linear):   0%|          | 0/1563 [00:00<?, ?it/s]

[ZeroS (Zero-Sum Linear)] Точность на тестовой выборке: 85.37%



Тестируем Softmax (Quadratic Baseline):   0%|          | 0/1563 [00:00<?, ?it/s]

[Softmax (Quadratic Baseline)] Точность на тестовой выборке: 86.64%



Тестируем Katharopoulos (Positive Linear):   0%|          | 0/1563 [00:00<?, ?it/s]

[Katharopoulos (Positive Linear)] Точность на тестовой выборке: 87.02%



### Интерпретация результатов
Выразительность и переобучение: Архитектура ZeroS продемонстрировала самую высокую точность на обучающей выборке, но уступила базовым моделям на тестовой. Знакопеременные веса с нулевой суммой действительно расширили математическую емкость и выразительность слоя внимания. Однако в контексте классификации текстов (IMDB) эта избыточная гибкость привела к переобучению: модель стала запоминать шум, в то время как строгие ограничения на положительность весов у Softmax и Katharopoulos сработали как естественный регуляризатор, обеспечив лучшую генерализацию.

Вычислительная эффективность (Speed): Несмотря на теоретическую линейную сложность $O(N)$, практическая скорость ZeroS оказалась в несколько раз ниже квадратичного $O(N^2)$ бейзлайна. Вероятно, это произошло из-за недостаточной оптимизации алгоритма ZeroS под железо. В то же время классический Softmax опирается на глубоко оптимизированные на уровне железа матричные умножения (fused-ядра/FlashAttention).

### Вывод
Метод ZeroS математически успешно доказывает, что линейному вниманию можно вернуть экспрессивность квадратичного путем удаления константы нулевого порядка. Тем не менее, на данный момент практическое применение этой архитектуры ограничено двумя факторами: склонностью к переобучению на задачах, не требующих сложного контрастивного анализа длинного контекста, и отсутствием низкоуровневых аппаратных оптимизаций (кастомных CUDA/Triton ядер). Для стандартных задач классификации текстов аппаратно-оптимизированный классический Softmax остается более надежным и быстрым.